# dim_position.py

## Purpose
Builds `dim_position` as an SCD1 dimension with all EA FC positions.

## Source
- Bronze: `fifa.bronze.ea_fc_players_raw` (column `position`)
- Reference: hardcoded dictionary `POSITIONS`

## Strategy
- SCD1 (positions are stable)
- Overwrite mode
- The dictionary includes historical positions (LWB, RWB) for compatibility with older FIFA editions
- Left join by position code to detect unmapped values

## Output
- Table: `fifa.silver.dim_position`

## Columns
- `position_id` (INT)
- `position_code` (STRING): GK, CB, ST...
- `position_name` (STRING): Goalkeeper, Center Back...
- `position_group` (STRING): Goalkeeper, Defender, Midfielder, Forward
- `_ingestion_timestamp` (TIMESTAMP)

## Why a hardcoded dictionary?

Positions don't change often, and EA uses a fixed set of codes (GK, CB, ST...). Hardcoding the dictionary:

1. Guarantees stable IDs across runs.
2. Allows including historical positions (LWB, RWB) that may not appear in the current dataset but existed in older editions.
3. Makes the mapping explicit and easy to review.


In [0]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql import Row
from pyspark.sql.types import IntegerType


# --- Hardcoded positions dictionary ---
POSITIONS = {
    "GK":  {"id": 1,  "name": "Goalkeeper",                   "group": "Goalkeeper"},
    "CB":  {"id": 2,  "name": "Center Back",                  "group": "Defender"},
    "LB":  {"id": 3,  "name": "Left Back",                    "group": "Defender"},
    "RB":  {"id": 4,  "name": "Right Back",                   "group": "Defender"},
    "LWB": {"id": 5,  "name": "Left Wing Back",               "group": "Defender"},
    "RWB": {"id": 6,  "name": "Right Wing Back",              "group": "Defender"},
    "CDM": {"id": 7,  "name": "Central Defensive Midfielder", "group": "Midfielder"},
    "CM":  {"id": 8,  "name": "Central Midfielder",           "group": "Midfielder"},
    "CAM": {"id": 9,  "name": "Central Attacking Midfielder", "group": "Midfielder"},
    "LM":  {"id": 10, "name": "Left Midfielder",              "group": "Midfielder"},
    "RM":  {"id": 11, "name": "Right Midfielder",             "group": "Midfielder"},
    "LW":  {"id": 12, "name": "Left Winger",                  "group": "Forward"},
    "RW":  {"id": 13, "name": "Right Winger",                 "group": "Forward"},
    "CF":  {"id": 14, "name": "Center Forward",               "group": "Forward"},
    "ST":  {"id": 15, "name": "Striker",                      "group": "Forward"},
}

# ===== Read Bronze =====
bronze = spark.read.table("fifa.bronze.ea_fc_players_raw")

# Convert the dictionary into a reference DataFrame
# Iterate over the dictionary and create a list of Rows.
# For "GK", it generates:
# Row(position_code="GK", position_id=1, position_name="Goalkeeper", position_group="Goalkeeper")
positions_rows = [
    Row(position_code=k, position_id=v["id"], position_name=v["name"], position_group=v["group"])
    for k, v in POSITIONS.items()
]

# Create the reference DataFrame from the list of Rows
dim_position_ref = spark.createDataFrame(positions_rows)

# Join with unique position codes from Bronze
dim_position = (
    bronze
    .select(col("position").alias("position_code"))
    .distinct()
    .join(dim_position_ref, on="position_code", how="left")
    .withColumn("position_id", col("position_id").cast(IntegerType())) 
    .withColumn("_ingestion_timestamp", current_timestamp())
    .select("position_id", "position_code", "position_name", "position_group", "_ingestion_timestamp")
    .orderBy("position_id")
)

# ===== Create table Silver =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.dim_position (
    position_id INT,
    position_code STRING,
    position_name STRING,
    position_group STRING,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA;
""")

# Overwrite the table (small catalog)
(dim_position
 .write
 .mode("overwrite")
 .option("mergeSchema", "true")
 .saveAsTable("fifa.silver.dim_position")
)

print(f"dim_position creada con {spark.read.table('fifa.silver.dim_position').count()} filas")

LIMIT 10 of the dim_position table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
select * from fifa.silver.dim_position limit 10;